# Project 2. Poker Hand Prediction by Classification

3장/4장 공개 카드로 최종 패 점수(0~9)를 예측하는 분류기를 학습하고, 예측 확률로 기대 수익을 계산해 콜/폴드를 결정한다.

- 전처리: 원본 (문양, 숫자) 대신 포커 점수를 결정하는 특징(같은 숫자 최대 개수, 페어 수, 플러시 가능성, 스트레이트 가능 구간 수, 로열 플러시 가능성)을 192차원 one-hot으로 변환
- 데이터 증강: 각 5장 패에서 가능한 모든 3장 조합(10개), 4장 조합(5개)을 학습 샘플로 사용
- 분류기: 다항 LogisticRegression (원본 특징 LogisticRegression / DecisionTree 와 비교)
- 필요 파일: poker-hand-training.csv, poker-hand-testing.csv (같은 폴더)

## 1. 특징 추출(전처리) 및 학습 함수

In [ ]:
# Project 2. Poker Hand Prediction by Classification - train.py
#
# 3장/4장의 카드만 보고 "최종 5장 패의 점수(0~9)"를 예측하는 분류기를 학습하고
# poker-model3.sav, poker-model4.sav 로 저장한다.
#
# 아이디어
#  1) 전처리(feature engineering): (문양, 숫자) 원본 값은 순서/크기에 의미가 없어서
#     분류기가 패턴을 배우기 어렵다. 대신 포커 점수를 결정하는 성질을 특징으로 만든다.
#       - 같은 숫자가 최대 몇 장인지 (max_rank_count: 1~4)
#       - 페어의 개수 (n_pairs: 0~2)
#       - 공개된 카드가 모두 같은 문양인지 (flush_live: 플러시 가능성)
#       - 스트레이트가 가능한 5칸 구간의 개수 (n_str_windows: 0~3, 에이스는 1과 14 모두 고려)
#       - 로열 플러시가 가능한지 (royal_live)
#     이 조합을 one-hot 벡터(192차원)로 만들어 분류기에 넣는다.
#  2) 데이터 증강: 카드가 공개되는 순서는 무작위이므로, 학습 데이터의 각 5장 패에서
#     가능한 모든 3장 조합(10개)과 4장 조합(5개)을 학습 샘플로 사용한다.
#  3) 분류기: 다항 로지스틱 회귀(LogisticRegression)로 P(최종 점수 = k | 공개된 카드)를 추정한다.
#     (원본 특징을 그대로 쓴 로지스틱 회귀/의사결정나무와 비교)
#  4) 베팅 결정(poker.py): 예측 확률로 이길 확률/질 확률을 구하고 기대 수익이 큰 쪽을 선택한다.

import itertools
import pickle

import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, log_loss

N_STATES = 4 * 3 * 2 * 4 * 2   # 192


def hand_features(suits, ranks):
    """suits, ranks: (N, n) 배열 (n = 3 또는 4). 반환: (N, 192) one-hot 특징 행렬"""
    suits = np.asarray(suits, dtype=int)
    ranks = np.asarray(ranks, dtype=int)
    if suits.ndim == 1:
        suits = suits.reshape(1, -1)
        ranks = ranks.reshape(1, -1)
    N, n = ranks.shape
    rows = np.arange(N)

    # 숫자별 개수 (인덱스 1~13, 14는 에이스를 하이로 볼 때)
    rc = np.zeros((N, 15), dtype=int)
    for i in range(N):          # 손패마다
        for j in range(n):      # 카드마다
            rc[i, ranks[i, j]] += 1
    rc[:, 14] = rc[:, 1]
    # 문양별 개수
    sc = np.zeros((N, 5), dtype=int)
    for i in range(N):          # 손패마다
        for j in range(n):      # 카드마다
            sc[i, suits[i, j]] += 1

    counts = rc[:, 1:14]
    max_rank = counts.max(axis=1)                       # 1~4
    n_pairs = (counts == 2).sum(axis=1)                 # 0~2
    distinct = (counts > 0).sum(axis=1)
    flush_live = (sc.max(axis=1) == n).astype(int)      # 0/1

    present = rc > 0
    n_str = np.zeros(N, dtype=int)
    royal_window = np.zeros(N, dtype=bool)
    for s in range(1, 11):                               # 구간 s ~ s+4 (A-2-3-4-5 ~ 10-J-Q-K-A)
        fits = (present[:, s:s + 5].sum(axis=1) == distinct) & (distinct == n)
        n_str += fits
        if s == 10:
            royal_window = fits
    n_str = np.minimum(n_str, 3)                         # 0~3
    royal_live = (royal_window & (flush_live == 1)).astype(int)

    state = ((((max_rank - 1) * 3 + n_pairs) * 2 + flush_live) * 4 + n_str) * 2 + royal_live
    X = np.zeros((N, N_STATES))
    X[rows, state] = 1.0
    return X


def raw_to_arrays(df, idx):
    """df의 카드 컬럼 중 idx(카드 번호 리스트)만 골라 suits, ranks 배열 반환"""
    v = df.iloc[:, :10].to_numpy()
    suits = np.stack([v[:, 2 * i] for i in idx], axis=1)
    ranks = np.stack([v[:, 2 * i + 1] for i in idx], axis=1)
    return suits, ranks


def build_dataset(df, n_cards, all_subsets=True):
    """각 5장 패에서 n_cards장 조합으로 (특징, 최종 점수) 샘플을 만든 뒤, 같은 상태끼리 묶어 가중치로 집계"""
    y = df.iloc[:, -1].to_numpy()
    subsets = list(itertools.combinations(range(5), n_cards)) if all_subsets else [tuple(range(n_cards))]
    states, labels = [], []
    for idx in subsets:
        s, r = raw_to_arrays(df, idx)
        states.append(hand_features(s, r).argmax(axis=1))
        labels.append(y)
    states = np.concatenate(states)
    labels = np.concatenate(labels)
    # (상태, 점수) 별 빈도를 sample_weight 로 사용 -> 수백만 샘플을 빠르게 학습
    agg = pd.DataFrame({'state': states, 'y': labels}).value_counts().reset_index()
    agg.columns = ['state', 'y', 'w']
    X = np.zeros((len(agg), N_STATES))
    X[np.arange(len(agg)), agg['state'].to_numpy()] = 1.0
    return X, agg['y'].to_numpy(), agg['w'].to_numpy().astype(float)


def train_model(df, n_cards):
    X, y, w = build_dataset(df, n_cards)
    # 기본 tol(1e-4)로는 최적화가 일찍 멈춰서 표본이 적은 상태(예: 4장이 모두 같은 숫자)의
    # 확률을 제대로 학습하지 못했다. tol 을 1e-8 로 낮춰 끝까지 수렴시킨다.
    model = LogisticRegression(C=100.0, max_iter=20000, tol=1e-8)
    model.fit(X, y, sample_weight=w)
    return model


def evaluate_probs(model, df, n_cards, featurize=True):
    s, r = raw_to_arrays(df, range(n_cards))
    y = df.iloc[:, -1].to_numpy()
    if featurize:
        X = hand_features(s, r)
    else:
        X = df.iloc[:, :2 * n_cards].to_numpy()
    p = model.predict_proba(X)
    pred = model.classes_[p.argmax(axis=1)]
    # 테스트에만 있는 클래스가 있을 수 있어 labels 지정
    full = np.zeros((len(y), 10))
    full[:, model.classes_] = p
    full = np.clip(full, 1e-12, 1)
    full /= full.sum(axis=1, keepdims=True)
    return accuracy_score(y, pred), log_loss(y, full, labels=list(range(10)))

## 2. 모델 비교, 학습 및 저장 (poker-model3.sav, poker-model4.sav)

In [ ]:
train = pd.read_csv('poker-hand-training.csv')
test = pd.read_csv('poker-hand-testing.csv')
train = train.apply(pd.to_numeric, errors='coerce').dropna().astype(int)
test = test.apply(pd.to_numeric, errors='coerce').dropna().astype(int)
print('train:', train.shape, ' test:', test.shape)

sub_tr = train.sample(50000, random_state=0)
for n in (3, 4):
    print('\n===== %d장 공개 모델 =====' % n)
    # (비교 1) 원본 특징 + 로지스틱 회귀 (샘플 코드 방식)
    base = LogisticRegression(max_iter=500)
    base.fit(sub_tr.iloc[:, :2 * n], sub_tr.iloc[:, -1])
    acc, ll = evaluate_probs(base, test, n, featurize=False)
    print('[원본 특징 + LogisticRegression]  accuracy=%.4f  log_loss=%.4f' % (acc, ll))
    # (비교 2) 원본 특징 + 의사결정나무
    tree = DecisionTreeClassifier(max_depth=10, random_state=0)
    tree.fit(sub_tr.iloc[:, :2 * n], sub_tr.iloc[:, -1])
    acc, ll = evaluate_probs(tree, test, n, featurize=False)
    print('[원본 특징 + DecisionTree]        accuracy=%.4f  log_loss=%.4f' % (acc, ll))
    # (최종) 전처리 특징 + 데이터 증강 + 로지스틱 회귀
    model = train_model(train, n)
    acc, ll = evaluate_probs(model, test, n)
    print('[전처리 특징 + LogisticRegression] accuracy=%.4f  log_loss=%.4f' % (acc, ll))
    filename = 'poker-model%d.sav' % n
    pickle.dump(model, open(filename, 'wb'))
    print('saved ->', filename)

train: (275010, 11)  test: (250000, 11)

===== 3장 공개 모델 =====
[원본 특징 + LogisticRegression]  accuracy=0.5011  log_loss=0.9833
[원본 특징 + DecisionTree]        accuracy=0.5421  log_loss=1.4071
[전처리 특징 + LogisticRegression] accuracy=0.6302  log_loss=0.8093
saved -> poker-model3.sav

===== 4장 공개 모델 =====
[원본 특징 + LogisticRegression]  accuracy=0.5011  log_loss=0.9835
[원본 특징 + DecisionTree]        accuracy=0.5541  log_loss=1.3747
[전처리 특징 + LogisticRegression] accuracy=0.7739  log_loss=0.5664
saved -> poker-model4.sav

## 3. 평가 (poker-eval.py 와 동일: 1,000게임 x 10회, 시작 금액 $1,000)

아래 셀의 FILL HERE 부분이 poker.py 에 들어간 코드와 동일하다.

In [ ]:
## ---------------- FILL HERE ---------------------------------
import pickle
import os
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression

# ------------------------------------------------------------------
# 전처리: 공개된 카드(3장 또는 4장)를 포커 점수와 관련된 특징으로 변환
#   - max_rank_count : 같은 숫자 카드의 최대 개수 (1~4)
#   - n_pairs        : 페어 개수 (0~2)
#   - flush_live     : 공개된 카드가 모두 같은 문양인지 (플러시 가능성)
#   - n_str_windows  : 스트레이트가 가능한 5칸 구간 수 (0~3, 에이스는 1/14 모두 고려)
#   - royal_live     : 로열 플러시 가능 여부
# 위 조합을 192차원 one-hot 벡터로 만든다. (train.py 와 동일한 함수)
# ------------------------------------------------------------------
N_STATES = 4 * 3 * 2 * 4 * 2


def hand_features(suits, ranks):
    suits = np.asarray(suits, dtype=int)
    ranks = np.asarray(ranks, dtype=int)
    if suits.ndim == 1:
        suits = suits.reshape(1, -1)
        ranks = ranks.reshape(1, -1)
    N, n = ranks.shape
    rows = np.arange(N)
    rc = np.zeros((N, 15), dtype=int)
    for i in range(N):          # 손패마다
        for j in range(n):      # 카드마다
            rc[i, ranks[i, j]] += 1
    rc[:, 14] = rc[:, 1]
    sc = np.zeros((N, 5), dtype=int)
    for i in range(N):          # 손패마다
        for j in range(n):      # 카드마다
            sc[i, suits[i, j]] += 1
    counts = rc[:, 1:14]
    max_rank = counts.max(axis=1)
    n_pairs = (counts == 2).sum(axis=1)
    distinct = (counts > 0).sum(axis=1)
    flush_live = (sc.max(axis=1) == n).astype(int)
    present = rc > 0
    n_str = np.zeros(N, dtype=int)
    royal_window = np.zeros(N, dtype=bool)
    for s in range(1, 11):
        fits = (present[:, s:s + 5].sum(axis=1) == distinct) & (distinct == n)
        n_str += fits
        if s == 10:
            royal_window = fits
    n_str = np.minimum(n_str, 3)
    royal_live = (royal_window & (flush_live == 1)).astype(int)
    state = ((((max_rank - 1) * 3 + n_pairs) * 2 + flush_live) * 4 + n_str) * 2 + royal_live
    X = np.zeros((N, N_STATES))
    X[rows, state] = 1.0
    return X


# load the model from disk
#  (sklearn 버전 차이 등으로 sav 로딩이 실패하면 train.py 로 즉석에서 다시 학습)
def _load_models():
    try:
        m3 = pickle.load(open('poker-model3.sav', 'rb'))
        m4 = pickle.load(open('poker-model4.sav', 'rb'))
        return m3, m4
    except Exception as e:
        print('model load failed (%s); retraining from poker-hand-training.csv' % e)
        from train import train_model
        df = pd.read_csv('poker-hand-training.csv')
        df = df.apply(pd.to_numeric, errors='coerce').dropna().astype(int)
        return train_model(df, 3), train_model(df, 4)


model3, model4 = _load_models()

# 52장 전체 덱 (문양 1~4, 숫자 1~13)
FULL_DECK = [(s, r) for s in range(1, 5) for r in range(1, 14)]


def _win_lose_prob(model, proba, opposite_score):
    classes = model.classes_
    p_win = proba[:, classes > opposite_score].sum(axis=1)
    p_lose = proba[:, classes < opposite_score].sum(axis=1)
    return p_win, p_lose


# given parameters:
#   hand: list of tuples (suit, rank); length is 3 or 4
#   opposite_score: the score of opposite player
#
# return:
#   True if you want to call,
#   False if you want to fold
#
# 기대 수익 기준 결정 (판돈 $1씩, 승리 시 +$3 / 패배 시 -$3 / 무승부 $0)
#   - 4장: 콜하면 3*(P승 - P패), 폴드하면 -2  ->  3*(P승 - P패) > -2 이면 콜
#   - 3장: 폴드하면 -1. 콜하면 다음 카드(남은 49장 중 하나)를 보고 다시 최선의 선택을 하므로
#          기대값 = 평균_{다음 카드} max(-2, 3*(P승 - P패))  ->  이 값이 -1 보다 크면 콜
def predict_call(hand, opposite_score):
    suits = [int(s) for s, r in hand]
    ranks = [int(r) for s, r in hand]

    if len(hand) == 4:
        proba = model4.predict_proba(hand_features(suits, ranks))
        p_win, p_lose = _win_lose_prob(model4, proba, opposite_score)
        return bool(3 * (p_win[0] - p_lose[0]) > -2)

    # 3장: 가능한 4번째 카드를 모두 고려
    known = set(zip(suits, ranks))
    nexts = [c for c in FULL_DECK if c not in known]
    S = np.array([suits + [c[0]] for c in nexts])
    R = np.array([ranks + [c[1]] for c in nexts])
    proba = model4.predict_proba(hand_features(S, R))
    p_win, p_lose = _win_lose_prob(model4, proba, opposite_score)
    value_if_call = np.maximum(-2, 3 * (p_win - p_lose)).mean()
    return bool(value_if_call > -1)
## -------------------------------------------------

import pickle
from sklearn.linear_model import LogisticRegression
import pandas as pd
import numpy as np
import random

test = pd.read_csv('poker-hand-testing.csv')

money = [1000] * 10
for i in range(10):
    samp = np.random.randint(0, len(test), size=1000)
    X_test = test.iloc[samp, :-1].to_numpy()
    y_test = test.iloc[samp, -1].to_numpy()

    for test_hand, test_score in zip(X_test, y_test):
        test_input = [ (suit, rank) for suit, rank in zip(test_hand[::2], test_hand[1::2]) ]
        oppo_score = y_test[random.randrange(len(y_test))]

        if not predict_call(test_input[:3], oppo_score):
            money[i] -= 1
            continue

        if not predict_call(test_input[:4], oppo_score):
            money[i] -= 2
            continue

        if test_score > oppo_score:
            money[i] += 3

        elif test_score < oppo_score:
            money[i] -= 3
        # if tie, your money stays the same

print('average of money= {}'.format(round(np.mean(money), 3)))

average of money= 1381.3

## 4. 전략별 비교 (같은 게임 10,000판)

| 전략 | 평균 최종 금액 |
|-|-|
| 항상 콜 | $1,005.7 |
| 샘플 코드 방식 (원본 특징 LogisticRegression) | $1,246.5 |
| 본 모델 (전처리 특징 + LogisticRegression + 기대값 판단) | **$1,384.0** |